# Aether Studio OmniVoice Service

Notebook này chạy OmniVoice TTS service trên Colab/GPU và expose endpoint public để Aether Studio local gọi qua HTTP.

Flow:
1. Kiểm tra GPU.
2. Clone/pull repo Aether Studio.
3. Cài dependencies.
4. Chuẩn bị `Voice_Ref.WAV`, `voice_scripts.txt`, `Instruction.txt`.
5. Start FastAPI service.
6. Expose bằng Cloudflare Tunnel.
7. Test `/health`, `/ready`, `/synthesize/file`.


In [ ]:
# 1. GPU check
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Runtime > Change runtime type > GPU')


In [ ]:
# 2. Clone hoặc update repo
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/kaisye/aether-studio.git'
PROJECT_DIR = Path('/content/aether-studio')

if PROJECT_DIR.exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull'], check=False)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)

SERVICE_DIR = PROJECT_DIR / 'omnivoice'
print('Service dir:', SERVICE_DIR)
print('Files:', [p.name for p in SERVICE_DIR.iterdir()])


In [ ]:
# 3. Install dependencies
%cd /content/aether-studio/omnivoice
!pip install -q -r requirements.txt


## 4. Chuẩn bị voice reference

Vì `Voice_Ref.WAV` là dữ liệu giọng nói cá nhân, file này không được push lên GitHub mặc định.

Nếu notebook báo thiếu `Voice_Ref.WAV`, chạy cell upload bên dưới và chọn file `Voice_Ref.WAV` từ máy local.

In [ ]:
# Upload Voice_Ref.WAV nếu file chưa có trong repo runtime
from pathlib import Path
from google.colab import files
import shutil

SERVICE_DIR = Path('/content/aether-studio/omnivoice')
ref_audio = SERVICE_DIR / 'Voice_Ref.WAV'

if not ref_audio.exists():
    print('Upload Voice_Ref.WAV')
    uploaded = files.upload()
    for name in uploaded:
        if name.lower().endswith(('.wav', '.mp3', '.flac')):
            shutil.move(name, ref_audio)
            break
else:
    print('Voice reference exists:', ref_audio)

print('Voice ref exists:', ref_audio.exists(), ref_audio)


In [ ]:
# Kiểm tra text reference và instruction. Có thể sửa nội dung tại đây nếu cần.
from pathlib import Path

SERVICE_DIR = Path('/content/aether-studio/omnivoice')
script_path = SERVICE_DIR / 'voice_scripts.txt'
instruction_path = SERVICE_DIR / 'Instruction.txt'

if not script_path.exists():
    script_path.write_text('Xin chao, day la cau noi tham chieu cho giong mau.', encoding='utf-8')
if not instruction_path.exists():
    instruction_path.write_text('male, Vietnamese, natural narrator, clear studio voice', encoding='utf-8')

print('voice_scripts.txt:', script_path.read_text(encoding='utf-8', errors='ignore')[:500])
print('Instruction.txt:', instruction_path.read_text(encoding='utf-8', errors='ignore')[:500])


In [ ]:
# 5. Start OmniVoice service trong background
import os, sys, time, subprocess
from pathlib import Path

SERVICE_DIR = Path('/content/aether-studio/omnivoice')
os.environ['OMNIVOICE_SHARE_PROVIDER'] = 'cloudflared'
os.environ['OMNIVOICE_LOAD_ASR'] = '0'
os.environ['OMNIVOICE_MODE'] = 'clone'
os.environ['OMNIVOICE_DEFAULT_REF_AUDIO_PATH'] = str(SERVICE_DIR / 'Voice_Ref.WAV')
os.environ['OMNIVOICE_DEFAULT_REF_TEXT_PATH'] = str(SERVICE_DIR / 'voice_scripts.txt')
os.environ['OMNIVOICE_DEFAULT_INSTRUCTION_PATH'] = str(SERVICE_DIR / 'Instruction.txt')

log_path = Path('/content/omnivoice_service.log')
cmd = [sys.executable, str(SERVICE_DIR / 'omnivoice_service.py')]
log_file = log_path.open('w')
service_proc = subprocess.Popen(cmd, cwd=str(SERVICE_DIR), stdout=log_file, stderr=subprocess.STDOUT)
print('Service PID:', service_proc.pid)
print('Log:', log_path)
time.sleep(5)
print(log_path.read_text(errors='ignore')[-2000:])


In [ ]:
# 6. Theo dõi health cho đến khi model ready
import requests, time

for i in range(60):
    try:
        data = requests.get('http://127.0.0.1:8008/health', timeout=10).json()
        print(i, data)
        if data.get('status') == 'ready' or data.get('model_loaded') is True:
            break
    except Exception as exc:
        print(i, 'not up yet:', exc)
    time.sleep(10)


In [ ]:
# 7. Test generate audio nội bộ Colab
import requests
from IPython.display import Audio, display

res = requests.post(
    'http://127.0.0.1:8008/synthesize/file',
    json={'text': 'Xin chao, day la ban thu nghiem giong noi tu OmniVoice.', 'mode': 'clone'},
    timeout=300,
)
print(res.status_code, res.headers.get('content-type'))
print(res.text[:500] if res.status_code >= 400 else 'audio ok')
if res.status_code == 200:
    out = '/content/omnivoice_test.wav'
    open(out, 'wb').write(res.content)
    display(Audio(out))


In [ ]:
# 8. Expose public URL bằng Cloudflare Tunnel
import re, subprocess, time, os
from pathlib import Path

cloudflared = Path('/content/cloudflared')
if not cloudflared.exists():
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /content/cloudflared

tunnel_log = Path('/content/cloudflared.log')
tunnel_file = tunnel_log.open('w')
tunnel_proc = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8008'],
    stdout=tunnel_file,
    stderr=subprocess.STDOUT,
)
print('Tunnel PID:', tunnel_proc.pid)

public_url = None
for _ in range(45):
    text = tunnel_log.read_text(errors='ignore') if tunnel_log.exists() else ''
    match = re.search(r'https://[a-zA-Z0-9.-]+\.trycloudflare\.com', text)
    if match:
        public_url = match.group(0)
        break
    time.sleep(1)

print('PUBLIC_URL=', public_url)
if public_url:
    print('Set local Aether .env:')
    print(f'AETHER_TTS_PROVIDER=omnivoice')
    print(f'OMNIVOICE_API_URL={public_url}')
    print(f'OMNIVOICE_MODE=clone')
else:
    print(tunnel_log.read_text(errors='ignore')[-4000:])


In [ ]:
# 9. Test public URL
import requests

assert public_url, 'Run the Cloudflare Tunnel cell first.'
print(requests.get(public_url + '/health', timeout=30).json())
res = requests.post(
    public_url + '/synthesize/file',
    json={'text': 'Day la audio test qua public tunnel.', 'mode': 'clone'},
    timeout=300,
)
print(res.status_code, res.headers.get('content-type'), len(res.content))


## Cấu hình Aether Studio local

Dán URL public vào `.env` trên máy Windows:

```env
AETHER_TTS_PROVIDER=omnivoice
OMNIVOICE_API_URL=https://xxxxx.trycloudflare.com
OMNIVOICE_API_KEY=
OMNIVOICE_MODE=clone
```

Sau đó restart backend Aether và chạy job video.